# Part 3 — QLoRA 통합 실습 (EXAONE 4.0 1.2B)
### 최신 EXAONE 4.0 + HuggingFace 공식 지원 버전

> **환경**: 윈도우 로컬 VSCode + venv + RTX 3080  
> **모델**: `LGAI-EXAONE/EXAONE-4.0-1.2B` (1.2B 소형 모델, 약 2.5GB)  
> **요구사항**: `transformers >= 4.54.0`

---

## ⚡ EXAONE 4.0의 장점

| 항목 | EXAONE 3.5 2.4B | **EXAONE 4.0 1.2B** |
|---|---|---|
| 크기 | 2.4B | **1.2B (절반)** |
| 한국어 | ✅ | ✅ |
| HF 공식 지원 | ❌ (커스텀 코드) | **✅ (공식 아키텍처)** |
| 호환성 패치 | 필요 | **불필요** |
| `trust_remote_code` | 필수 | **불필요** |
| VRAM (4-bit) | ~2GB | **~1GB** |
| 학습 속도 | 느림 | **빠름 (~2배)** |

→ **설정 훨씬 간단 + 학습 빠름** (실습 최적)

---

## 🎯 학습 목표

| # | 미션 |
|---|------|
| 1 | transformers 4.54+ 설치 |
| 2 | HuggingFace 로그인 (.env) |
| 3 | EXAONE 4.0 1.2B 4-bit 로드 (패치 없음!) |
| 4 | LoRA 어댑터 부착 |
| 5 | 베이스라인 응답 5건 수집 + 저장 |

---
## 0️⃣ 환경 준비 — 최신 transformers 설치

**이론 요약**
- EXAONE 4.0은 `transformers >= 4.54.0` 필요
- 2025년 7월 26일 공식 지원 추가됨
- `trust_remote_code` 불필요 (공식 아키텍처)

### 미션 0-1. 패키지 설치 (최신 버전)

In [1]:
%pip install -q --upgrade transformers accelerate peft bitsandbytes datasets openai python-dotenv huggingface_hub

Note: you may need to restart the kernel to use updated packages.


> ⚠️ **설치 완료 후 반드시 커널 재시작!**  
> 상단 Kernel → Restart Kernel

### 미션 0-2. 버전 확인 (커널 재시작 후 실행)

In [2]:
import sys
import torch
import transformers, peft, accelerate

print(f"Python:       {sys.executable}")
print(f"PyTorch:      {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")

# EXAONE 4.0 필수 조건 체크
최소버전 = "4.54.0"
현재버전 = transformers.__version__
if tuple(map(int, 현재버전.split('.')[:2])) >= (4, 54):
    print(f"\n✅ transformers {현재버전} >= {최소버전} → EXAONE 4.0 사용 가능!")
else:
    print(f"\n❌ transformers {현재버전} < {최소버전} → 업그레이드 필요")
    print("  %pip install --upgrade transformers 실행 후 커널 재시작")

Python:       c:\ai_workspace\코드파일\venv\Scripts\python.exe
PyTorch:      2.5.1+cu121
transformers: 5.13.0
peft:         0.19.1
accelerate:   1.14.0
GPU:          NVIDIA GeForce RTX 3080

✅ transformers 5.13.0 >= 4.54.0 → EXAONE 4.0 사용 가능!


✅ **결과 확인**: transformers 4.54+ 이고 ✅ 표시 나오나요?  
👉 체크 실패 시 아래 셀에서 강제 업그레이드.

### 미션 0-3. (필요 시) 강제 업그레이드

In [2]:
# 미션 0-2에서 ❌가 나온 경우에만 실행
# %pip install --upgrade --force-reinstall transformers>=4.54.0
# → 실행 후 커널 재시작 → 0-2 재실행
print("이 셀은 필요할 때만 주석 해제 후 실행하세요")

이 셀은 필요할 때만 주석 해제 후 실행하세요


### 미션 0-4. VRAM 확인 함수

In [3]:
import gc

def print_gpu_memory(라벨=""):
    사용 = torch.cuda.memory_allocated() / 1e9
    전체 = torch.cuda.get_device_properties(0).total_memory / 1e9
    바 = "█" * int(사용 / 전체 * 30)
    빈 = "░" * (30 - len(바))
    print(f"[{라벨}] {사용:.2f}GB / {전체:.1f}GB  [{바}{빈}] {사용/전체*100:.0f}%")

print_gpu_memory("시작")

[시작] 0.00GB / 10.7GB  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0%


---
## 1️⃣ HuggingFace 로그인

**이론 요약**
- EXAONE 4.0도 Access Request 필요
- `.env`의 `HUGGINGFACE_TOKEN` 사용

> ⚠️ 사전 준비: [EXAONE-4.0-1.2B](https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B) 페이지에서 Access Request 승인받기.

In [4]:
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv()

token = os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise ValueError("❌ .env에 HUGGINGFACE_TOKEN이 없습니다")

login(token=token)
print(f"✅ HuggingFace 로그인 완료")

✅ HuggingFace 로그인 완료


---
## 2️⃣ CONFIG 설정

**이론 요약**
- EXAONE 4.0 1.2B는 hidden_size 2048 (3.5 2.4B는 2560)
- r=16이면 적절 (모델이 작아서 r=32는 과함)
- alpha = r × 2 = 32

In [5]:
CONFIG = {
    "model_name": "LGAI-EXAONE/EXAONE-4.0-1.2B",
    "max_seq_length": 1024,
    "lora_r": 16,           # EXAONE 4.0 1.2B에 맞게 조정
    "lora_alpha": 32,       # r × 2
    "lora_dropout": 0.05,
    "seed": 42,
}

print("📋 CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

📋 CONFIG:
  model_name: LGAI-EXAONE/EXAONE-4.0-1.2B
  max_seq_length: 1024
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  seed: 42


---
## 3️⃣ 토크나이저 + 양자화 설정

**이론 요약**
- EXAONE 4.0은 `trust_remote_code` **불필요** (공식 지원!)
- `compute_dtype=bfloat16` 권장 (RTX 3080 네이티브 지원)
- 패딩 설정은 동일

### 미션 3-1. 토크나이저 로드

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
# 👆 trust_remote_code 없음! EXAONE 4.0은 공식 지원이라 불필요

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

print(f"✅ 토크나이저 로드 완료")
print(f"   어휘 크기: {tokenizer.vocab_size:,}")
print(f"   pad_token: {tokenizer.pad_token}")

✅ 토크나이저 로드 완료
   어휘 크기: 102,400
   pad_token: [PAD]


### 미션 3-2. BitsAndBytesConfig

In [7]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # EXAONE 4.0은 bfloat16 권장
    bnb_4bit_use_double_quant=True,
)

print("✅ 양자화 설정 완료 (compute_dtype=bfloat16)")

✅ 양자화 설정 완료 (compute_dtype=bfloat16)


---
## 4️⃣ EXAONE 4.0 1.2B 로드 (패치 없음!)

**이론 요약**
- EXAONE 3.5에서 필요했던 `get_input_embeddings` 패치 **불필요**
- `attn_implementation="eager"` **불필요**
- `trust_remote_code=True` **불필요**
- **HF 공식 아키텍처라 그냥 로드하면 끝**

```
첫 다운로드 예상:
  1.2B 모델 = 약 2.5GB
  다운로드: 2~4분 (네트워크에 따라)
  4-bit 양자화 후 VRAM: ~1GB
```

💡 **왜 이렇게 간단한가?**  
EXAONE 4.0은 2025년 7월 26일 HuggingFace transformers에 **공식 통합**됐습니다. 더 이상 커스텀 코드나 패치가 필요 없어요.

### 미션 4-1. 모델 로드

In [8]:
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training

print("🔄 EXAONE 4.0 1.2B 4-bit 로드 중... (2~4분)")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    # trust_remote_code 없음! attn_implementation 없음!
)

model = prepare_model_for_kbit_training(model)

print(f"\n✅ 모델 로드 완료")
print_gpu_memory("EXAONE 4.0 1.2B 4-bit 로드 후")

🔄 EXAONE 4.0 1.2B 4-bit 로드 중... (2~4분)


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]


✅ 모델 로드 완료
[EXAONE 4.0 1.2B 4-bit 로드 후] 1.39GB / 10.7GB  [███░░░░░░░░░░░░░░░░░░░░░░░░░░░] 13%


✅ **결과 확인**: 약 **1GB** 정도 사용되나요?  
👉 에러 없이 깔끔하게 로드! 더 이상 패치 필요 없음.

---
## 5️⃣ 모델 구조 확인 (target_modules 결정용)

**이론 요약**
- EXAONE 4.0의 실제 레이어 이름 확인
- Llama 계열 구조 → `q_proj, k_proj, v_proj, o_proj`
- Full 버전(3.5)의 `out_proj`와 다름

### 미션 5-1. 실제 Linear 레이어 이름 확인

In [9]:
linear_modules = set()
for name, module in model.named_modules():
    if "Linear" in type(module).__name__ or "4bit" in type(module).__name__:
        if len(name.split(".")) >= 2:
            linear_modules.add(name.split(".")[-1])

print("📋 EXAONE 4.0 1.2B의 Linear 레이어 이름:")
for 이름 in sorted(linear_modules):
    print(f"  - {이름}")

📋 EXAONE 4.0 1.2B의 Linear 레이어 이름:
  - down_proj
  - gate_proj
  - k_proj
  - o_proj
  - q_proj
  - up_proj
  - v_proj


✅ **결과 확인**: 어떤 이름들이 보이나요?  
👉 예상: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj, lm_head`

---
## 6️⃣ LoRA 어댑터 부착 → QLoRA 완성

**이론 요약**
- EXAONE 4.0은 Llama 계열 구조 → `q_proj, k_proj, v_proj, o_proj`
- MLP까지 포함: `gate_proj, up_proj, down_proj`
- r=16 (1.2B 모델에 적절)

### 미션 6-1. LoraConfig + get_peft_model

In [10]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # Attention
        "gate_proj", "up_proj", "down_proj",         # MLP
    ],
)

model = get_peft_model(model, lora_config)
print("🚀 QLoRA 완성!")

🚀 QLoRA 완성!


### 미션 6-2. 학습 파라미터 확인

In [11]:
model.print_trainable_parameters()

trainable params: 15,237,120 || all params: 1,294,628,608 || trainable%: 1.1769


✅ **결과 확인**: `trainable%` 가 **약 0.5~1.5%** 로 나왔나요?  
👉 EXAONE 4.0 1.2B 기준 약 800만~2000만 개 학습.

### 미션 6-3. 최종 메모리 확인

In [12]:
print_gpu_memory("QLoRA 완성 후")

여유 = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"\n남은 VRAM: {여유:.1f}GB → Part 4 학습 충분 ✅")

[QLoRA 완성 후] 1.45GB / 10.7GB  [████░░░░░░░░░░░░░░░░░░░░░░░░░░] 14%

남은 VRAM: 9.3GB → Part 4 학습 충분 ✅


---
## 7️⃣ 베이스라인 응답 수집

**이론 요약**
- EXAONE 4.0은 `apply_chat_template` 표준 방식 사용
- 3.5의 `[|system|]...[|endofturn|]` 포맷과 다름
- HuggingFace 공식 채팅 템플릿 자동 적용

### 미션 7-1. 추론 함수 (EXAONE 4.0 표준 방식)

In [13]:
def generate_fast(model, tokenizer, question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": "당신은 대한민국 법률 전문 AI 어시스턴트입니다. 관련 법조항과 법률 원칙에 근거하여 정확한 답변을 제공합니다."},
        {"role": "user", "content": question}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False
    ).to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,            # EXAONE 4.0 1.2B 한국어 권장값
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # 입력 부분 제외하고 디코딩
    생성 = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(생성, skip_special_tokens=True).strip()

print("✅ 추론 함수 정의 완료")

✅ 추론 함수 정의 완료


> 💡 **EXAONE 4.0 1.2B 권장 파라미터** (공식 문서):  
> - `temperature=0.1` (한국어 코드 스위칭 방지)  
> - `top_p=0.9`

### 미션 7-2. 1개 질문 테스트

In [19]:
def generate_fast(model, tokenizer, question, max_new_tokens=256):
    prompt = f"""다음 질문에 대해 한국어로 간단하고 정확하게 답변하세요.

질문: {question}

답변:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # 입력 프롬프트 부분을 제외하고 새로 생성된 답변만 추출
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [24]:
TEST_QUESTIONS = [
    "임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?",
    "근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해야 하나요?",
    "온라인 쇼핑에서 환불을 거부당했을 때 소비자 권리는 무엇인가요?",
    "교통사고 후 보험사와 합의할 때 주의사항은 무엇인가요?",
    "이웃 간 소음 분쟁이 발생했을 때 법적 대응 방법은 무엇인가요?",
]

print(f"[질문] {TEST_QUESTIONS[0]}\n")
응답 = generate_fast(model, tokenizer, TEST_QUESTIONS[2])
print(f"[응답]\n{응답}")

[질문] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?

[응답]
소비자 권리를 포함한 정보를 포함한 답변.


✅ **결과 확인**: 한국어 법률 답변이 나오나요?  
👉 EXAONE 4.0 1.2B는 3.5 2.4B보다 작지만 Reasoning 기능으로 품질 유지.

### 미션 7-3. 5개 질문 전체 수집

In [16]:
print("🔄 베이스라인 응답 생성 중...\n")

baseline_results = []
for i, 질문 in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] {질문[:30]}...")
    응답 = generate_fast(model, tokenizer, 질문)
    baseline_results.append({
        "question": 질문,
        "baseline_answer": 응답,
    })

print(f"\n✅ {len(baseline_results)}건 수집 완료")

🔄 베이스라인 응답 생성 중...

[1/5] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요...
[2/5] 근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해...
[3/5] 온라인 쇼핑에서 환불을 거부당했을 때 소비자 권리는 무...
[4/5] 교통사고 후 보험사와 합의할 때 주의사항은 무엇인가요?...
[5/5] 이웃 간 소음 분쟁이 발생했을 때 법적 대응 방법은 무...

✅ 5건 수집 완료


---
## 8️⃣ 베이스라인 저장

In [26]:
import json

os.makedirs("./outputs", exist_ok=True)
저장경로 = "./outputs/baseline_results3.json"

with open(저장경로, "w", encoding="utf-8") as f:
    json.dump({
        "model": CONFIG["model_name"],
        "config": {
            "r": CONFIG["lora_r"],
            "lora_alpha": CONFIG["lora_alpha"],
            "target_modules": lora_config.target_modules
        },
        "results": baseline_results,
    }, f, ensure_ascii=False, indent=2,default=list)

print(f"✅ 저장 완료: {저장경로}")
print(f"   크기: {os.path.getsize(저장경로)/1024:.1f} KB")

✅ 저장 완료: ./outputs/baseline_results3.json
   크기: 2.3 KB


In [27]:
import json

os.makedirs("./outputs", exist_ok=True)
저장경로 = "./outputs/baseline_results4.json"

with open(저장경로, "w", encoding="utf-8") as f:
    json.dump({
        "model": CONFIG["model_name"],
        "config": {
            "r": CONFIG["lora_r"],
            "lora_alpha": CONFIG["lora_alpha"],
            "target_modules": lora_config.target_modules
        },
        "results": baseline_results,
    }, f, ensure_ascii=False, indent=2,default=list)

print(f"✅ 저장 완료: {저장경로}")
print(f"   크기: {os.path.getsize(저장경로)/1024:.1f} KB")

✅ 저장 완료: ./outputs/baseline_results4.json
   크기: 2.3 KB


---
## 🎓 Part 3 완료 요약

In [35]:
print("=" * 55)
print("📋 Part 3 완료 상태 (EXAONE 4.0 1.2B)")
print("=" * 55)
print(f"✅ 모델:          {CONFIG['model_name']}")
print(f"✅ 양자화:         4-bit NF4 + Double Quant (bfloat16)")
print(f"✅ LoRA:          r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}")
print(f"✅ target:        Attention + MLP (7개 레이어)")
print(f"✅ 패치:          불필요 (HF 공식 지원)")
print(f"✅ 베이스라인:     {len(baseline_results)}건 저장")
print()
print_gpu_memory("현재")
print()
print("⏭️ Part 4 예고: 데이터 로드 → SFTTrainer 학습!")

📋 Part 3 완료 상태 (EXAONE 4.0 1.2B)
✅ 모델:          LGAI-EXAONE/EXAONE-4.0-1.2B
✅ 양자화:         4-bit NF4 + Double Quant (bfloat16)
✅ LoRA:          r=8, alpha=16
✅ target:        Attention + MLP (7개 레이어)
✅ 패치:          불필요 (HF 공식 지원)
✅ 베이스라인:     5건 저장

[현재] 1.46GB / 10.7GB  [████░░░░░░░░░░░░░░░░░░░░░░░░░░] 14%

⏭️ Part 4 예고: 데이터 로드 → SFTTrainer 학습!


In [28]:
from datasets import load_dataset
import random

ds = load_dataset("jihye-moon/LawQA-Ko", split="train")

# 컬럼 이름 확인
q_col, a_col = ds.column_names[:2]
print(f"컬럼: 질문='{q_col}', 답변='{a_col}'")

# 200건 샘플링
random.seed(42)
ds = ds.select(random.sample(range(len(ds)), 200))
print(f"✅ 학습 데이터: {len(ds)}건")

컬럼: 질문='question', 답변='precedent'
✅ 학습 데이터: 200건


In [29]:
SYSTEM = "당신은 대한민국 법률 전문 AI 어시스턴트입니다."

def preprocess(sample):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": sample[q_col]},
        {"role": "assistant", "content": sample[a_col]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized = ds.map(preprocess, remove_columns=ds.column_names)
print(f"✅ 토크나이징 완료: {len(tokenized)}건")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ 토크나이징 완료: 200건


In [57]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./exaone4-law-quick",
    num_train_epochs=3,                        # 2 에폭
    per_device_train_batch_size=2,             # 배치 2
    gradient_accumulation_steps=4,             # 실효 배치 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",                        # 체크포인트 저장 안함 (속도)
    report_to="none",
    seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)
예상스텝 = len(tokenized) // 8 * 2
print(f"✅ 예상 스텝: 약 {예상스텝}")

✅ 예상 스텝: 약 50


In [58]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

print("🏋️ 학습 시작!\n")
result = trainer.train()

print(f"\n✅ 학습 완료!")
print(f"   train_loss: {result.training_loss:.4f}")
print(f"   소요 시간:  {result.metrics['train_runtime']:.0f}초")

🏋️ 학습 시작!



c:\ai_workspace\코드파일\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.278994
10,2.100436
15,2.177272
20,2.021429
25,2.234017
30,1.736933
35,1.916942
40,1.656572
45,1.517225
50,1.641592



✅ 학습 완료!
   train_loss: 1.7524
   소요 시간:  117초


In [59]:
질문 = TEST_QUESTIONS[0]

In [60]:
print(질문)

임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?


In [80]:
model.eval()

# 첫 질문으로 Before/After 비교
질문 = TEST_QUESTIONS[1]
학습후 = generate_fast(model, tokenizer, 질문)
베이스라인 = baseline_results[1]["baseline_answer"]

print(f"[질문] {질문}\n")
print("=" * 55)
print("[학습 전]")
print("=" * 55)
print(베이스라인[:300])
print("\n" + "=" * 55)
print("[학습 후]")
print("=" * 55)
print(학습후[:300])

[질문] 근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해야 하나요?

[학습 전]
근로계약서를 만들고 그 사이에 있는 사람을 기준으로 한 정보를 포함하여 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포

[학습 후]
질문: 근로계약서 없음 + 일한 경우 + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + +


In [68]:
# 5개 질문 전체 수집
finetuned_results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] 생성 중...")
    finetuned_results.append({
        "question": q,
        "baseline": baseline_results[i-1]["baseline_answer"],
        "finetuned_raw": generate_fast(model, tokenizer, q),
    })

print(f"\n✅ {len(finetuned_results)}건 완료")

[1/5] 생성 중...
[2/5] 생성 중...
[3/5] 생성 중...
[4/5] 생성 중...
[5/5] 생성 중...

✅ 5건 완료


In [69]:
# 어댑터 + 결과 저장
import json, os

trainer.save_model("./exaone4-law-raw")
tokenizer.save_pretrained("./exaone4-law-raw")

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/finetuned_raw_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_loss": result.training_loss,
        "results": finetuned_results,
    }, f, ensure_ascii=False, indent=2)

print("✅ 어댑터 저장: ./exaone4-law-raw/")
print("✅ 결과 저장:   ./outputs/finetuned_raw_results.json")

✅ 어댑터 저장: ./exaone4-law-raw/
✅ 결과 저장:   ./outputs/finetuned_raw_results.json


In [85]:
from openai import OpenAI
from datasets import load_dataset
import json, random

client = OpenAI()  # .env의 OPENAI_API_KEY 자동 사용

# 원본 데이터 로드
ds_orig = load_dataset("jihye-moon/LawQA-Ko", split="train")
q_col, a_col = ds_orig.column_names[:2]

random.seed(42)
indices = random.sample(range(len(ds_orig)), 200)

print(f"✅ 정제할 샘플: {len(indices)}건")

✅ 정제할 샘플: 200건


In [86]:
def refine(question, answer):
    prompt = f"""당신은 대한민국 법률 교육 전문가입니다.
아래 법률 상담 QA를 **법률 원칙 학습용 데이터**로 정제해주세요.

[원본 질문]
{question[:600]}

[원본 답변]
{answer[:1000]}

[정제 규칙]
1. 질문에서 구체적 인명, 지명, 날짜, 금액을 제거하고 법률 원칙을 묻는 일반적 질문으로.
2. 답변에서 특정 사건 사실관계를 제거하고 법조항·법률 원칙 중심으로 재작성.
3. 답변은 300~500자. 반드시 한국어.

JSON 형식으로만 출력: {{"question": "...", "answer": "..."}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=800, temperature=0.3,
        )
        text = resp.choices[0].message.content.strip()
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except:
        return None

# 1건 테스트
s = ds_orig[indices[0]]
test_result = refine(s[q_col], s[a_col])
print("📋 정제 테스트:")
print(f"  원본 Q: {s[q_col][:100]}...")
print(f"  정제 Q: {test_result['question'] if test_result else '실패'}")

📋 정제 테스트:
  원본 Q: 주택 임대차 계약을 체결하려고 하는데, 계약 당일에 주택의 소유자가 바빠서 소유자의 부인이랑 임대차 계약을 체결하라고 합니다. 부인이랑 임대차 계약을 체결해도 괜찮을까요?...
  정제 Q: 주택 임대차 계약을 체결할 때, 소유자가 아닌 제3자와 계약을 체결하는 것이 가능한가요?


In [87]:
import time

# 200건 일괄 정제 (약 5분)
refined = []
failed = 0

print("🔄 정제 시작...\n")
for i, idx in enumerate(indices):
    s = ds_orig[idx]
    r = refine(s[q_col], s[a_col])
    
    if r and r.get("question") and r.get("answer"):
        refined.append({"instruction": r["question"], "output": r["answer"]})
    else:
        failed += 1
    
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/200] 성공 {len(refined)} / 실패 {failed}")
    time.sleep(0.3)

print(f"\n✅ 정제 완료: {len(refined)}건")

🔄 정제 시작...

  [20/200] 성공 20 / 실패 0
  [40/200] 성공 40 / 실패 0
  [60/200] 성공 60 / 실패 0
  [80/200] 성공 80 / 실패 0
  [100/200] 성공 100 / 실패 0
  [120/200] 성공 120 / 실패 0
  [140/200] 성공 140 / 실패 0
  [160/200] 성공 160 / 실패 0
  [180/200] 성공 180 / 실패 0
  [200/200] 성공 200 / 실패 0

✅ 정제 완료: 200건


In [88]:
# 정제 데이터 저장
import os
os.makedirs("./outputs", exist_ok=True)

with open("./outputs/refined_data.json", "w", encoding="utf-8") as f:
    json.dump(refined, f, ensure_ascii=False, indent=2)

print(f"💾 저장: ./outputs/refined_data.json ({len(refined)}건)")
print(f"\n📋 정제 샘플 2개:")
for i, d in enumerate(refined[:2]):
    print(f"\n[{i+1}] Q: {d['instruction'][:100]}")
    print(f"    A: {d['output'][:150]}...")

💾 저장: ./outputs/refined_data.json (200건)

📋 정제 샘플 2개:

[1] Q: 주택 임대차 계약을 체결할 때, 소유자가 아닌 다른 사람과 계약을 체결해도 괜찮은가요?
    A: 주택 임대차 계약은 원칙적으로 임대인과 임차인 간의 합의에 의해 성립합니다. 임대인은 자신의 권리를 다른 사람에게 위임할 수 있으며, 이 경우 위임을 받은 사람이 계약을 체결할 수 있습니다. 그러나 임대인의 부인이 계약을 체결할 경우, 임대인의 명시적인 위임이 필요합니...

[2] Q: 자동차를 운전하는 동안 사고가 발생했을 때, 운전자가 아닌 차량 소유자에게도 책임이 인정될 수 있는지 궁금합니다. 특히, 차량 소유자가 운전자를 지정하여 차량을 운전하게 한 경우에
    A: 자동차손해배상보장법에 따르면, 자동차의 운행으로 인한 사고에 대해 운전자의 책임이 인정됩니다. 그러나 차량 소유자가 운전자를 지정하여 차량을 운전하게 한 경우, 소유자에게도 일정 부분 책임이 인정될 수 있습니다. 이는 '운행자성'의 개념에 기초하며, 차량 소유자가 운전...


In [98]:
# 기존 모델 해제
import gc
del model
gc.collect()
import torch
torch.cuda.empty_cache()

print("✅ Part 4 model 해제 완료")
print(f"현재 VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

✅ Part 4 model 해제 완료
현재 VRAM: 0.86GB


In [99]:
# Part 3처럼 EXAONE 4.0 + QLoRA 다시 로드
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

print("🔄 EXAONE 4.0 재로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], quantization_config=bnb_config, device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
print("✅ QLoRA 재적용 완료")
model.print_trainable_parameters()

🔄 EXAONE 4.0 재로드 중...


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

✅ QLoRA 재적용 완료
trainable params: 15,237,120 || all params: 1,294,628,608 || trainable%: 1.1769


In [100]:
# 정제 데이터로 Dataset 변환 + 토크나이징
from datasets import Dataset

SYSTEM = "당신은 대한민국 법률 전문 AI 어시스턴트입니다."

def preprocess_refined(s):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": s["instruction"]},
        {"role": "assistant", "content": s["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

ds_refined = Dataset.from_list(refined)
tokenized_refined = ds_refined.map(preprocess_refined, remove_columns=ds_refined.column_names)
print(f"✅ 토크나이징: {len(tokenized_refined)}건")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ 토크나이징: 200건


In [101]:
# 학습 실행 (50스텝)
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./exaone4-law-refined-tmp",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)

trainer = Trainer(model=model, args=args, train_dataset=tokenized_refined, data_collator=collator)

print("🏋️ 정제 데이터 학습 시작!\n")
result_refined = trainer.train()
print(f"\n✅ 완료! train_loss: {result_refined.training_loss:.4f}")

🏋️ 정제 데이터 학습 시작!



c:\ai_workspace\코드파일\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.408968
10,1.521348
15,1.301342
20,1.173119
25,1.189663
30,1.020230
35,0.993106
40,0.994966
45,0.962994
50,0.951167



✅ 완료! train_loss: 1.2517


In [102]:
# 정제 학습본으로 5개 질문 응답 생성
model.eval()

refined_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] 생성 중...")
    refined_answers.append(generate_fast(model, tokenizer, q))

print("✅ 정제 학습본 응답 5건 완료")

[1/5] 생성 중...
[2/5] 생성 중...
[3/5] 생성 중...
[4/5] 생성 중...
[5/5] 생성 중...
✅ 정제 학습본 응답 5건 완료


In [104]:
# 3가지 비교: 베이스라인 vs 원본학습 vs 정제학습
질문 = TEST_QUESTIONS[1]
베이스라인 = baseline_results[1]["baseline_answer"]
원본학습 = finetuned_results[1]["finetuned_raw"]
정제학습 = refined_answers[1]

print(f"[질문] {질문}\n")
for 이름, 응답 in [("1️⃣ 베이스라인 (학습 없음)", 베이스라인),
                  ("2️⃣ 원본 데이터 학습", 원본학습),
                  ("3️⃣ 정제 데이터 학습", 정제학습)]:
    print("=" * 55)
    print(이름)
    print("=" * 55)
    print(응답[:300])
    print()

[질문] 근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해야 하나요?

1️⃣ 베이스라인 (학습 없음)
근로계약서를 만들고 그 사이에 있는 사람을 기준으로 한 정보를 포함하여 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포함하게 되어 있는 정보를 포

2️⃣ 원본 데이터 학습
질문: 근로계약서 없음 + 일한 경우 + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + + +

3️⃣ 정제 데이터 학습
근로계약서가 없다는 사실을 고려하여, 임금 체불에 대한 대처가 필요하다는 사실을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 필요하다는 점을 포함하여, 이러한 요소들이 



In [105]:
def judge(question, answer):
    prompt = f"""당신은 대한민국 법률 전문가입니다.

[질문]
{question}

[답변]
{answer[:800]}

4가지 기준 각 1~10점. JSON만 출력:
{{"법률_정확성": N, "핵심_포함": N, "오류_여부": N, "실용성": N, "총점": N}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=0,
        )
        text = resp.choices[0].message.content.strip()
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except:
        return None

# 5개 질문 × 3개 모델 = 15번 채점
judge_results = []
print("🧑‍⚖️ Judge 채점 중...\n")

for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  [{i}/5] {q[:30]}...")
    j_base = judge(q, baseline_results[i-1]["baseline_answer"])
    j_raw = judge(q, finetuned_results[i-1]["finetuned_raw"])
    j_ref = judge(q, refined_answers[i-1])
    judge_results.append({
        "question": q, "base": j_base, "raw": j_raw, "refined": j_ref,
    })
    time.sleep(0.5)

print("\n✅ 채점 완료")

🧑‍⚖️ Judge 채점 중...

  [1/5] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요...
  [2/5] 근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해...
  [3/5] 온라인 쇼핑에서 환불을 거부당했을 때 소비자 권리는 무...
  [4/5] 교통사고 후 보험사와 합의할 때 주의사항은 무엇인가요?...
  [5/5] 이웃 간 소음 분쟁이 발생했을 때 법적 대응 방법은 무...

✅ 채점 완료


In [106]:
# 평균 점수 계산
def 평균(key):
    vals = [r[key]["총점"] for r in judge_results if r[key]]
    return sum(vals) / len(vals) if vals else 0

base_avg = 평균("base")
raw_avg = 평균("raw")
refined_avg = 평균("refined")

print("=" * 55)
print("📊 최종 비교 (총점 40 만점)")
print("=" * 55)
print(f"1️⃣ 베이스라인:      {base_avg:.1f} / 40")
print(f"2️⃣ 원본 데이터 학습: {raw_avg:.1f} / 40  ({raw_avg-base_avg:+.1f})")
print(f"3️⃣ 정제 데이터 학습: {refined_avg:.1f} / 40  ({refined_avg-base_avg:+.1f})")
print()

if refined_avg > raw_avg + 1:
    print(f"✅ 정제 학습이 원본 학습보다 {refined_avg-raw_avg:+.1f}점 우수!")
    print("   → 데이터 품질이 모델 성능을 결정한다는 핵심 교훈 확인")
elif raw_avg > refined_avg + 1:
    print(f"🔵 원본 학습이 유리 → 정제 데이터 품질/양 보강 필요")
else:
    print(f"🟡 비슷한 수준 → 더 많은 데이터 또는 에폭 필요")

📊 최종 비교 (총점 40 만점)
1️⃣ 베이스라인:      29.4 / 40
2️⃣ 원본 데이터 학습: 30.6 / 40  (+1.2)
3️⃣ 정제 데이터 학습: 26.6 / 40  (-2.8)

🔵 원본 학습이 유리 → 정제 데이터 품질/양 보강 필요


In [107]:
# 결과 저장
comparison = {
    "baseline_avg": base_avg,
    "raw_trained_avg": raw_avg,
    "refined_trained_avg": refined_avg,
    "details": [
        {
            "question": r["question"],
            "base_score": r["base"]["총점"] if r["base"] else 0,
            "raw_score": r["raw"]["총점"] if r["raw"] else 0,
            "refined_score": r["refined"]["총점"] if r["refined"] else 0,
        }
        for r in judge_results
    ],
}

with open("./outputs/comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

# 어댑터 저장
trainer.save_model("./exaone4-law-refined")
tokenizer.save_pretrained("./exaone4-law-refined")

print("💾 저장 완료:")
print("  ./exaone4-law-refined/ (정제 학습 어댑터)")
print("  ./outputs/comparison_results.json (비교 결과)")

💾 저장 완료:
  ./exaone4-law-refined/ (정제 학습 어댑터)
  ./outputs/comparison_results.json (비교 결과)


---

## 🎯 EXAONE 3.5 → 4.0 전환의 장점

| 항목 | 3.5 (이전) | 4.0 (지금) |
|---|---|---|
| `trust_remote_code` | 필수 | **불필요** |
| `attn_implementation="eager"` | 필수 | **불필요** |
| `get_input_embeddings` 패치 | 필수 | **불필요** |
| `ATTENTION_FUNCTIONS` 패치 | 필수 | **불필요** |
| 프롬프트 포맷 | `[|system|]...[|endofturn|]` | `apply_chat_template` 표준 |
| `target_modules` | `out_proj, c_fc_*` | `o_proj, gate/up/down_proj` |
| 모델 크기 | 2.4B (~5GB) | **1.2B (~2.5GB)** |
| 학습 속도 | 기준 | **~2배 빠름** |

### 🔜 Part 4에서 할 일

1. LawQA-Ko 데이터 로드
2. EXAONE 4.0 `apply_chat_template`로 포매팅
3. 토크나이징 + train/eval split
4. `TrainingArguments` 설정 (`bf16=True`)
5. `Trainer.train()` 실행 (1.2B 모델이라 **10~20분** 예상)
6. 학습 후 응답 생성 → 베이스라인과 비교

**1.2B 모델 덕분에 학습이 3.5 2.4B 대비 훨씬 빠릅니다.** 🚀